# 分布式训练

> 以 7B 模型为例，AdamW 与混合精度训练需要约 112 GB 固定显存，已经超过一张 80 GB A100 的容量。多卡训练首先是一道拆分题：哪些状态必须保留，哪些状态可以分到不同设备。
>
> **单卡账本**：参数、梯度、FP32 master 权重，以及 AdamW 的一阶矩和二阶矩，共同构成固定显存开销。
>
> **状态切分**：ZeRO Stage 1、2、3 依次切分优化器状态、梯度与模型参数。
>
> **计算切分**：数据并行、张量并行与流水线并行组成 3D 并行，Megatron-LM 用这些维度安排多卡计算。
>
> **工程落地**：DeepSpeed 描述切分与卸载策略，Accelerate 统一训练脚本、有效批量大小和检查点恢复。

本节不涉及集合通信的底层实现（all-reduce 的 ring 算法），相关内容见附录《all_reduce 与集合通信》；张量并行与流水线并行的手算见附录《大模型的五种并行切分》。

In [ ]:
# 本章所有 import 集中放在第一个 code cell
import json
import os
import shutil

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

from accelerate import Accelerator

torch.manual_seed(42)
print("torch:", torch.__version__)

## 1. 单卡显存开销

AdamW 与混合精度训练下，每个参数占用 16 bytes 显存，由五部分组成：

| 状态 | 精度 | 大小 |
|:---|:---|:---|
| 参数 | FP16 | 2 bytes |
| 梯度 | FP16 | 2 bytes |
| master 权重 | FP32 | 4 bytes |
| AdamW 一阶矩 | FP32 | 4 bytes |
| AdamW 二阶矩 | FP32 | 4 bytes |

写成公式，其中 $P$ 为参数量：

$$16P = 2P + 2P + 4P + 4P + 4P$$

这部分开销与序列长度无关，仅由参数量决定。

In [ ]:
# === 单卡显存开销：7B 模型 ===
P = 7e9  # 7B 参数

param_fp16   = 2 * P   # FP16 参数
grad_fp16    = 2 * P   # FP16 梯度
master_fp32  = 4 * P   # FP32 master 权重
adam_m       = 4 * P   # AdamW 一阶矩
adam_v       = 4 * P   # AdamW 二阶矩

fixed_bytes = param_fp16 + grad_fp16 + master_fp32 + adam_m + adam_v
fixed_gb = fixed_bytes / 1e9

print(f"模型参数 (FP16):    {param_fp16 / 1e9:.1f} GB")
print(f"梯度 (FP16):        {grad_fp16 / 1e9:.1f} GB")
print(f"master 权重 (FP32): {master_fp32 / 1e9:.1f} GB")
print(f"AdamW m (FP32):     {adam_m / 1e9:.1f} GB")
print(f"AdamW v (FP32):     {adam_v / 1e9:.1f} GB")
print(f"固定开销合计:       {fixed_gb:.0f} GB (= 16 × P bytes)")

7B 参数的固定开销为 112 GB，其中优化器相关的三部分（master 权重和两个矩）合计 84 GB，是最大的部分。

多卡最直接的用法是数据并行（Distributed Data Parallel，DDP）：每张卡持有完整的模型副本，数据集切分为 N 份分给 N 张卡，各自完成前向和反向传播后，通过 all-reduce 将梯度平均。DDP 的吞吐量接近随卡数线性增长，但显存没有任何节省：8 张卡共 896 GB 显存中，有 7/8 是完全相同的冗余副本。

## 2. ZeRO 三阶段切分

ZeRO（Zero Redundancy Optimizer）将每张卡上冗余的 16P bytes 状态按块逐步切分到 N 张卡上，切分后的多卡在逻辑上仍等价于一个完整模型。三个阶段递进，每升一级多切分一块：

- **Stage 1** 切分优化器状态，12P 降为 12P/N，每张卡只更新自己负责的那 1/N 参数；
- **Stage 2** 进一步切分梯度，2P 降为 2P/N，反向传播时通过 reduce-scatter 让每张卡只保留自己那片梯度；
- **Stage 3** 进一步切分参数，2P 降为 2P/N，前向与反向传播时按需 all-gather 对应层，用完即释放。

In [ ]:
# === ZeRO 三阶段显存：7B 模型 × 8 卡 ===
P = 7e9
N = 8

print(f"{'方案':<16}{'参数':>8}{'梯度':>8}{'优化器':>8}{'单卡合计':>10}{'相对 DDP':>10}")
print("-" * 64)
configs = [
    ("DDP",          2 * P,       2 * P,       12 * P),
    ("ZeRO Stage 1", 2 * P,       2 * P,       12 * P / N),
    ("ZeRO Stage 2", 2 * P,       2 * P / N,   12 * P / N),
    ("ZeRO Stage 3", 2 * P / N,   2 * P / N,   12 * P / N),
]
ddp_bytes = 16 * P
for name, p, g, o in configs:
    total = p + g + o
    print(f"{name:<16}{p/1e9:>7.1f} {g/1e9:>7.1f} {o/1e9:>7.1f} "
          f"{total/1e9:>8.1f} GB {total/ddp_bytes*100:>8.1f}%")

从 DDP 到 Stage 3，单卡显存从 112 GB 降到 14 GB，压缩为原来的 1/8。代价是通信量递增：阶段越高，前向与反向传播需要传递的碎片越多。

工程实践中，Stage 2 的通信开销与 DDP 接近，是性价比最高的选择；参数确实放不下时才使用对节点间带宽更敏感的 Stage 3。

## 3. 3D 并行与 Megatron-LM

ZeRO 的两个主流实现是 DeepSpeed 和 PyTorch FSDP，它们本质上是数据并行的变体。当模型规模达到几百亿、集群达到几千卡时，这类方案会遇到瓶颈：ZeRO-3 每层前向都需要 all-gather 参数，跨节点通信量随规模增长而快速膨胀。

从零预训练大模型的工业标准是 Megatron-LM 的 3D 并行，即三种并行方式的组合：

- **张量并行（Tensor Parallelism，TP）** 将每一层的矩阵乘法按维度切分到多张卡。通信为每层一次 all-reduce，对延迟敏感，因此只部署在节点内部、利用 NVLink 带宽；
- **流水线并行（Pipeline Parallelism，PP）** 将模型按层切分为若干段，不同段部署在不同节点上接力执行。通信量小（只传输段边界的激活），适合跨节点的高延迟环境；
- **数据并行（Data Parallelism，DP）** 在最外层复制若干份模型副本，用 ZeRO 切分冗余状态。

部署时的经验法则是：TP 限制在节点内，PP 跨节点，剩余的卡分配给 DP。

Megatron 与 DeepSpeed / FSDP 的定位不同：后者是给数据并行训练提供显存优化，Megatron 是一整套从零预训练的框架，模型定义、数据加载、loss 计算和 checkpoint 都已内置。它的常见参数如下：

| 参数 | 含义 |
|:---|:---|
| `--tensor-model-parallel-size 8` | TP 度 = 8，每层矩阵乘切 8 份，锁在节点内 8 卡 |
| `--pipeline-model-parallel-size 16` | PP 度 = 16，模型按层切 16 段，跨节点接力 |
| `--global-batch-size 1024` | 一个完整 step 的样本数，框架自动推算梯度累积步数 |
| `--sequence-parallel` | 序列维也切分，和 TP 配合省激活显存 |
| `--recompute-activations` | 重算激活（梯度检查点），预训练省显存标配 |

三个典型场景的工具选择：

| 场景 | 工具 |
|:---|:---|
| 从零预训练 70B+、几千卡 | Megatron 系（3D 并行） |
| 从零预训练 7B~13B | Accelerate + FSDP 也够用 |
| 微调、几百卡以内 | Accelerate + ZeRO / FSDP |

（TP / PP 的内部原理和手算见附录《大模型的五种并行切分》；PyTorch 官方的 torchtitan 采用 FSDP + TP 的轻量路线，适合中等规模。）

## 4. DeepSpeed 的 ZeRO 配置

显存 OOM 后的排查顺序，按代价从小到大排列：

1. 减小 micro-batch、增大梯度累积；
2. ZeRO Stage 2 升级到 Stage 3；
3. 开启 `offload_optimizer`，将优化器状态卸载到 CPU 内存（节省最多，但 CPU↔GPU 搬运有开销）；
4. 开启 `offload_param`，将参数也卸载到 CPU（仅 Stage 3 支持）；
5. 使用 NVMe 硬盘卸载（ZeRO-Infinity）。

这些配置项写在 DeepSpeed 的 JSON 配置文件里，下一节的 Accelerate 通过 `--ds_config_file` 传入同一份文件：

In [ ]:
# === DeepSpeed ZeRO 配置：最常调整的几个字段 ===
deepspeed_config = {
    "train_micro_batch_size_per_gpu": 4,
    "bf16": {"enabled": True},
    "zero_optimization": {
        "stage": 3,                    # 1 切优化器状态 / 2 再切梯度 / 3 再切参数
        "offload_optimizer": {         # 优化器状态卸到 CPU 内存
            "device": "cpu",
            "pin_memory": True,        # 锁页内存，CPU→GPU 拷贝更快
        },
        "offload_param": {             # 参数也卸到 CPU（仅 stage 3 生效）
            "device": "none",
        },
        "overlap_comm": True,          # 通信和计算重叠，藏掉一部分通信延迟
        "contiguous_gradients": True,  # 梯度存成连续内存块，减少通信碎片
        "reduce_bucket_size": 5e8,     # 梯度分桶大小（bytes），大桶通信高效但耗显存
    },
}

print("ds_config.json:")
print(json.dumps(deepspeed_config, indent=2))

实际使用中经常调整的只有 stage 和两个 offload，其余保持默认即可：

| 参数 | 作用 | 调整时机 |
|:---|:---|:---|
| `stage`（1/2/3） | 切优化器状态 / 梯度 / 参数 | 显存不够就升级，常用 2 和 3 |
| `offload_optimizer` | 优化器状态卸到 CPU | Stage 3 仍不够时，以速度换显存 |
| `offload_param` | 参数也卸到 CPU | 模型大到 GPU 放不下，仅 Stage 3 |
| `overlap_comm` | 通信与计算重叠 | 默认开启；多卡通信慢时确认它开着 |
| `contiguous_gradients` | 梯度连续存储 | 默认开启，一般不用管 |
| `reduce_bucket_size` | 梯度分桶大小（bytes） | 通信是瓶颈时可调大，代价是显存 |

如果后端选择 FSDP，概念与 DeepSpeed 一一对应：`FULL_SHARD` 等价于 ZeRO-3，`SHARD_GRAD_OP` 等价于 ZeRO-2。两者的选择更多是工程偏好：DeepSpeed 配置驱动、offload 生态更完整；FSDP 由 PyTorch 官方维护、对新硬件的支持更快。下一节的 Accelerate 可以在两者之间切换而无需修改代码。

## 5. 用 Accelerate 编写多卡训练脚本

本节将「完成第一次预训练与微调」中的迷你 Trainer 改写为多卡版本。直接使用 PyTorch 时，更换后端意味着修改代码：DDP 需要 `torchrun` 启动并用 DistributedSampler 包装 DataLoader，FSDP 需要用包装器处理模型，DeepSpeed 需要初始化 engine 并编写 JSON 配置。

Accelerate 将这些差异统一到一个 `Accelerator` 对象之后。它本身不是分布式算法：DDP 模式下底层是 torch.distributed，FSDP 模式下是 PyTorch FSDP，DeepSpeed 模式下是 DeepSpeed。

改写分五步：模型与数据、Accelerator 初始化、训练循环、检查点、启动方式。

### 5.1 模型与数据

模型是一个极小的因果语言模型：embedding 层接 lm_head 层。数据是若干带固定模式的 token 序列。虽然规模很小，但接口与真实模型一致：Dataset 提供单条样本，DataLoader 拼接 batch，模型接收 `input_ids` 和 `labels` 并返回 loss。这部分代码与单卡脚本完全相同，不需要任何分布式改造。

In [ ]:
# === 模型与数据：与单卡脚本完全相同 ===
class TinyCausalLM(nn.Module):
    """极小的因果语言模型，embedding 接 lm_head。"""

    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.lm_head = nn.Linear(hidden_size, vocab_size)

    def forward(self, input_ids, labels=None):
        """
        输入 token ids 输出 logits；给定 labels 时同时返回 loss。

        input_ids: [batch, seq_len]，去掉结尾 token 的样本
        labels:    [batch, seq_len]，左移一位的下一个 token
        """
        logits = self.lm_head(self.embedding(input_ids))
        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), labels.reshape(-1))
        return {"loss": loss, "logits": logits}


class ToyTextDataset(Dataset):
    """每条样本是一小段 token ids：[1] 开头 [2] 结尾，中间是固定模式。"""

    def __init__(self, sequences):
        self.sequences = sequences

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, index):
        ids = self.sequences[index]
        # 输入取前 n-1 个 token，监督信号是后 n-1 个（预测下一个 token）
        return {
            "input_ids": torch.tensor(ids[:-1]),
            "labels": torch.tensor(ids[1:]),
        }


def simple_collate(features):
    """把一批样本 dict 拼成整批 tensor。"""
    return {
        "input_ids": torch.stack([f["input_ids"] for f in features]),
        "labels": torch.stack([f["labels"] for f in features]),
    }


# 两种交替出现的固定模式，模型可以学到「1 后面是 3」这类规律
sequences = [
    [1, 3, 4, 5, 6, 2] if i % 2 == 0 else [1, 3, 4, 6, 5, 2]
    for i in range(16)
]

dataset = ToyTextDataset(sequences)
dataloader = DataLoader(dataset, batch_size=4, shuffle=False, collate_fn=simple_collate)

model = TinyCausalLM(vocab_size=9, hidden_size=32)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.05)

batch = next(iter(dataloader))
print("一个 batch 的形状：", {k: tuple(v.shape) for k, v in batch.items()})

输出显示 `input_ids` 和 `labels` 的形状均为 `[4, 5]`：4 个样本、每个 5 个 token。到这里为止，代码中没有任何分布式相关的内容。

### 5.2 初始化 Accelerator

多卡改造从创建 `Accelerator` 开始。`prepare` 方法接收训练三件套（model、optimizer、dataloader），根据当前后端完成包装。`mixed_precision` 在 GPU 上通常配置为 `"bf16"`，本节的 CPU 演示关闭混合精度：

In [ ]:
# === 创建 Accelerator 并 prepare ===
accelerator = Accelerator(
    gradient_accumulation_steps=2,  # 2 个 micro-batch 攒够，才算一次完整 step
    mixed_precision="no",           # CPU 演示不开混合精度；真卡上配 "bf16"
)

model, optimizer, dataloader = accelerator.prepare(model, optimizer, dataloader)

print("distributed_type:", accelerator.distributed_type)
print("num_processes:   ", accelerator.num_processes)
print("process_index:   ", accelerator.process_index)
print("is_main_process: ", accelerator.is_main_process)

单进程执行时，`num_processes` 为 1、`process_index` 为 0、`is_main_process` 为 True。使用 `accelerate launch --num_processes 8` 启动后，每个进程获得不同的 `process_index` 和不同的数据分片，而这份代码不需要任何修改。

### 5.3 训练循环

与单卡训练循环相比，改动共有三处：

- `with accelerator.accumulate(model)` 管理梯度累积：未攒够 2 个 micro-batch 时，块内的 `optimizer.step()` 不会真正更新参数；
- `accelerator.backward(loss)` 取代 `loss.backward()`，DDP 模式下完成跨卡梯度同步，DeepSpeed ZeRO 模式下负责梯度的切分与聚合；
- 日志输出由 `is_main_process` 控制，避免 N 个进程重复打印。

In [ ]:
# === 完整训练循环：可以原封不动放进 train.py ===
num_epochs = 8

for epoch in range(num_epochs):
    total_loss, num_steps = 0.0, 0

    for batch in dataloader:
        with accelerator.accumulate(model):
            outputs = model(batch["input_ids"], labels=batch["labels"])
            accelerator.backward(outputs["loss"])
            optimizer.step()
            optimizer.zero_grad()

        if accelerator.is_main_process:
            total_loss += outputs["loss"].item()
            num_steps += 1

    if accelerator.is_main_process and (epoch + 1) % 2 == 0:
        print(f"epoch {epoch + 1:02d} | train_loss = {total_loss / num_steps:.4f}")

loss 从约 0.65 下降到约 0.42（具体数值因随机初始化而略有不同），训练行为与单卡一致。同一份循环，单进程执行是单卡训练，8 进程启动即为 8 卡数据并行。

### 5.4 有效批量大小

`gradient_accumulation_steps` 与另外两个数共同决定每次参数更新使用的样本数：

$$\text{有效 batch} = \text{micro-batch} \times \text{GPU 数} \times \text{累积步数}$$

上一节的示例为 4 × 1 × 2 = 8。在实际项目中，micro-batch 4、8 卡、累积 4 步对应有效 batch 128：每张卡一次只需要容纳 4 个样本的激活，而参数更新基于 128 个样本的梯度。显存消耗由 micro-batch 决定，训练效果由有效 batch 决定。

另外两个常用启动参数在这里一并说明。`num_processes` 是 GPU 总数，每个进程运行同一份脚本并处理不同的数据分片。`mixed_precision` 目前工业界默认 `bf16`：A100/H100 原生支持，数值范围与 FP32 相同，无需 FP16 那套 loss scale（见附录《混合精度训练与 loss scaling》）。

### 5.5 检查点的保存与恢复

实际训练通常持续数天到数周，需要支持中断后继续训练。Accelerate 提供一对配套方法：`save_state` 将模型、优化器和 DataLoader 进度整体保存；`load_state` 在新的进程中恢复。这两个方法在每个进程上调用即可，多进程一致性由 Accelerate 保证。

In [ ]:
# === 检查点演示：保存 → 模拟新进程恢复 → 验证权重一致 ===
ckpt_dir = "_ckpt_demo"

accelerator.save_state(ckpt_dir)
print("save_state 保存的文件：")
for name in sorted(os.listdir(ckpt_dir)):
    print(" ", name)

# 模拟「换台机器从检查点恢复」：新进程 = 新的 Accelerator + 未训练的新模型
accelerator_restored = Accelerator(
    gradient_accumulation_steps=2,
    mixed_precision="no",
)
model_restored = TinyCausalLM(vocab_size=9, hidden_size=32)
optimizer_restored = torch.optim.AdamW(model_restored.parameters(), lr=0.05)
model_restored, optimizer_restored = accelerator_restored.prepare(
    model_restored, optimizer_restored
)
accelerator_restored.load_state(ckpt_dir)

w_trained = accelerator.unwrap_model(model).embedding.weight
w_restored = accelerator_restored.unwrap_model(model_restored).embedding.weight
assert torch.allclose(w_trained, w_restored), "恢复后的权重应与保存时一致"

shutil.rmtree(ckpt_dir)  # 演示完清理临时目录

`save_state` 保存了三类文件：模型权重、优化器状态和随机数状态。`load_state` 之后权重与保存时逐位一致，AdamW 的一阶矩和二阶矩也一并恢复，训练可以从检查点无缝继续。

### 5.6 启动方式

脚本完成后，切换后端只需要更换启动命令。以下命令中的 `train.py` 即前面几个小节代码的合集：

In [ ]:
# === 启动命令：单卡、多卡、DeepSpeed 用同一份脚本 ===
print("单进程（等价于 python train.py）：")
print("  $ accelerate launch train.py")
print()
print("单机 8 卡：")
print("  $ accelerate launch --num_processes 8 --multi_gpu train.py")
print()
print("DeepSpeed ZeRO-3 后端（搭配上一节的 ds_config.json）：")
print("  $ accelerate launch --use_deepspeed --ds_config_file ds_config.json \\")
print("      --num_processes 8 train.py")
print()
print("也可以先运行 accelerate config（交互式问答）生成配置文件，")
print("之后统一用 accelerate launch --config_file xxx.yaml train.py 启动。")

从 DDP 切换到 FSDP 或 DeepSpeed 时，修改的只有启动命令（或 `accelerate config` 生成的配置文件），5.1~5.5 的代码保持不变。

## 6. 微调常用配置

最后列举与分布式训练配合使用的高频配置，几乎所有微调脚本都会用到：

- **梯度累积**：显存放不下大 batch 时以时间换空间；
- **梯度检查点**（gradient checkpointing）：反向传播时不保存中间激活，需要时重算，激活显存减少 60% 以上，整体速度下降约 30%；
- **BF16 混合精度**：参数和激活使用 BF16 存储，计算速度提升约一倍、显存减少一半；
- **FlashAttention-2**：注意力计算不物化完整的 attention 矩阵，长序列场景下显存和速度同时受益（原理见附录《FlashAttention 的分块计算》）；
- **8-bit 优化器**：AdamW 状态从 12P 压缩到 3P bytes，精度损失可忽略。

HuggingFace 生态中的对应写法：

| 配置 | 解决的问题 | 典型写法 |
|:---|:---|:---|
| 梯度累积 | batch 大显存装不下 | `gradient_accumulation_steps=8` |
| 梯度检查点 | 激活值占用显存 | `gradient_checkpointing=True` |
| BF16 混合精度 | 计算速度与显存 | `bf16=True` |
| FlashAttention-2 | 长序列注意力慢且占显存 | `attn_implementation='flash_attention_2'` |
| 8-bit 优化器 | 优化器状态 12P 过大 | `optim='adamw_bnb_8bit'` |

典型的参数组合：7B 模型单卡 LoRA 微调使用 bf16 + 梯度累积 + FlashAttention-2；70B 模型多卡全量微调使用 bf16 + ZeRO-3 + offload + 梯度检查点。

## 小结

- 7B 全量训练的固定开销约为 112 GB，即 16 bytes × 参数量；DDP 多卡不节省这部分显存
- ZeRO 的三个阶段分别切分优化器状态、梯度和参数，越切越省，通信代价越高
- 3D 并行的经验法则：TP 限制在节点内，PP 跨节点，剩余的卡分配给 DP
- 显存 OOM 的排查顺序：梯度累积 → Stage 2 → Stage 3 → offload optimizer → offload param
- 多卡脚本相对单卡脚本的改动集中在四处：Accelerator、prepare、accumulate/backward、is_main_process
- 有效 batch = micro-batch × GPU 数 × 梯度累积步数
- 检查点使用 save_state / load_state，模型和优化器状态一并恢复
- 工具选择：从零预训练大模型用 Megatron 系，微调用 Accelerate + ZeRO/FSDP

## 作业

> 可以让 AI 帮忙解释思路，但不建议直接让 AI "做完这道题"。

**作业 1：计算 ZeRO Stage 2 在 4 卡下的单卡显存**

7B 模型，AdamW 训练，4 张卡。ZeRO Stage 2 下每张卡的固定显存是多少 GB？

小提示：Stage 2 切分梯度和优化器状态（14P 切成 4 份），参数仍每卡完整保留（2P）。

In [ ]:
# 作业 1：ZeRO Stage 2 在 4 卡下的单卡显存
P = 7e9
N = 4

# TODO: 计算 Stage 2 的单卡显存（单位 GB）
# 参数完整保留 + (梯度 + 优化器状态) 切 N 份
s2_per_card_gb = (2 * P + 14 * P / N) / 1e9

assert s2_per_card_gb is not None, "请先计算 Stage 2 单卡显存"
expected = (2 * P + 14 * P / N) / 1e9
assert abs(s2_per_card_gb - expected) < 0.1, f"应为 {expected:.1f} GB"
print(f"✅ 作业 1 通过：")
print(f"   Stage 2 + 4 卡 + 7B：单卡 {s2_per_card_gb:.1f} GB")
print(f"   相比 DDP 的 112 GB 省了 {112 - s2_per_card_gb:.1f} GB，通信代价却几乎没涨。")

**作业 2：编写一份显存告急时的 DeepSpeed 配置**

补全下面的 `deepspeed_config`，要求：ZeRO Stage 3、优化器状态 offload 到 CPU、开启通信计算重叠。

小提示：对应 `zero_optimization.stage`、`offload_optimizer.device`、`overlap_comm` 三个字段。

In [ ]:
# 作业 2：显存告急时的 DeepSpeed 配置
deepspeed_config = {
    "bf16": {"enabled": True},
    "zero_optimization": {
        "stage": 3,
        "offload_optimizer": {"device": "cpu"},
        "overlap_comm": True,
    },
}

# 验证
zero = deepspeed_config.get("zero_optimization", {})
assert zero.get("stage") == 3, "stage 应为 3"
assert zero.get("offload_optimizer", {}).get("device") == "cpu", "offload_optimizer.device 应为 'cpu'"
assert zero.get("overlap_comm") is True, "overlap_comm 应为 True"

print("✅ 作业 2 通过：")
print(json.dumps(deepspeed_config, indent=2))
print()
print("这份配置 = 显存最紧张时的第一档方案：参数/梯度/优化器全切 + 优化器上 CPU。")

**作业 3：计算有效 batch size**

一个预训练任务：micro-batch 2，32 张卡，梯度累积 8 步。一个完整优化 step 使用了多少个样本？

小提示：三个数相乘，这就是 5.4 节公式里三个因子的现实版本。

In [ ]:
# 作业 3：计算有效 batch size
micro_batch = 2
num_gpus = 32
grad_accum = 8

# TODO: 计算有效 batch size
effective_batch = micro_batch * num_gpus * grad_accum

assert effective_batch is not None, "请先计算有效 batch size"
expected = micro_batch * num_gpus * grad_accum
assert effective_batch == expected, f"应为 {expected}"
print(f"✅ 作业 3 通过：")
print(f"   有效 batch = {micro_batch} × {num_gpus} × {grad_accum} = {effective_batch}")
print(f"   单卡一次只需要装 {micro_batch} 个样本的激活，却达到了 {effective_batch} 的 batch 效果。")

## 参考资料

- Rajbhandari et al., [ZeRO: Memory Optimizations Toward Training Trillion Parameter Models](https://arxiv.org/abs/1910.02054), 2020
- [HuggingFace Accelerate 文档](https://huggingface.co/docs/accelerate/)
- [DeepSpeed ZeRO 配置项文档](https://www.deepspeed.ai/docs/config-json/)
- Shoeybi et al., [Megatron-LM: Training Multi-Billion Parameter Language Models Using Model Parallelism](https://arxiv.org/abs/1909.08053), 2019
- [NVIDIA Megatron-LM GitHub](https://github.com/NVIDIA/Megatron-LM)